# PlanCritic: Getting Started

This notebook provides an introduction to PlanCritic, a physics-informed trajectory evaluation framework for autonomous driving.\n
## Overview

PlanCritic combines neural network-based learning with physics-based constraints to evaluate trajectory quality. This tutorial will walk you through:

1. **Basic Setup**: Installing dependencies and loading data
2. **Data Loading**: Working with WOMD and synthetic datasets
3. **Model Usage**: Loading pre-trained critics and scoring trajectories
4. **Physics Analysis**: Understanding physics-based evaluation metrics
5. **Visualization**: Creating interactive plots and visualizations

Let's get started!

## 1. Setup and Imports

In [ ]:
# Install required packages (run once)
# !pip install -e ../.

import numpy as np
import matplotlib.pyplot as plt
import torch
import pandas as pd
from pathlib import Path
import json

# PlanCritic imports
from plancritic.models.critic import TrajectoryCritic
from plancritic.data.adapters import WOMDAdapter, SyntheticAdapter
from plancritic.eval.physics_checks import PhysicsChecker
from plancritic.eval.metrics import TrajectoryMetrics
from plancritic.maps.lanegraph import LaneGraph

# Set up plotting
plt.style.use('seaborn-v0_8')
plt.rcParams['figure.figsize'] = (12, 8)

print("✅ Setup complete!")

## 2. Generate Synthetic Data

For this tutorial, we'll create synthetic trajectory data to demonstrate the framework without requiring large datasets.

In [ ]:
def generate_synthetic_scene():
    """Generate a synthetic driving scene with multiple trajectory candidates."""
    
    # Scene parameters
    scene_length = 100.0  # meters
    scene_width = 20.0    # meters
    num_lanes = 3
    lane_width = scene_width / num_lanes
    
    # Generate lane graph
    lane_centers = []
    for i in range(num_lanes):
        y_center = (i + 0.5) * lane_width - scene_width / 2
        lane_centers.append(y_center)
    
    # Generate ego vehicle trajectory candidates
    time_horizon = 8.0  # seconds
    dt = 0.1
    timesteps = int(time_horizon / dt)
    
    candidates = []
    
    # Candidate 1: Straight trajectory (good)
    straight_traj = np.zeros((timesteps, 4))  # [x, y, vx, vy]
    for t in range(timesteps):
        straight_traj[t, 0] = 15.0 * t * dt  # constant velocity
        straight_traj[t, 1] = 0.0  # center lane
        straight_traj[t, 2] = 15.0  # 15 m/s velocity
        straight_traj[t, 3] = 0.0
    candidates.append(('straight', straight_traj))
    
    # Candidate 2: Lane change trajectory (moderate)
    lane_change_traj = np.zeros((timesteps, 4))
    for t in range(timesteps):
        progress = t / timesteps
        lane_change_traj[t, 0] = 15.0 * t * dt
        # Smooth lane change using sigmoid
        lane_change_traj[t, 1] = lane_centers[1] * (1 / (1 + np.exp(-10 * (progress - 0.5))))
        lane_change_traj[t, 2] = 15.0
        # Lateral velocity during lane change
        if 0.3 < progress < 0.7:
            lane_change_traj[t, 3] = 2.0 * np.sin(np.pi * (progress - 0.3) / 0.4)
    candidates.append(('lane_change', lane_change_traj))
    
    # Candidate 3: Aggressive trajectory (poor)
    aggressive_traj = np.zeros((timesteps, 4))
    for t in range(timesteps):
        # High acceleration and jerk
        aggressive_traj[t, 0] = 10.0 * t * dt + 2.0 * (t * dt) ** 2
        aggressive_traj[t, 1] = 2.0 * np.sin(0.5 * t * dt)  # weaving
        aggressive_traj[t, 2] = 10.0 + 4.0 * t * dt  # accelerating
        aggressive_traj[t, 3] = 1.0 * np.cos(0.5 * t * dt)  # lateral oscillation
    candidates.append(('aggressive', aggressive_traj))
    
    # Generate other agents (for collision checking)
    other_agents = []
    # Agent in adjacent lane
    agent1_traj = np.zeros((timesteps, 4))
    for t in range(timesteps):
        agent1_traj[t, 0] = 50.0 + 12.0 * t * dt  # slightly ahead, slower
        agent1_traj[t, 1] = lane_centers[1]  # right lane
        agent1_traj[t, 2] = 12.0
        agent1_traj[t, 3] = 0.0
    other_agents.append(agent1_traj)
    
    return {
        'candidates': candidates,
        'other_agents': other_agents,
        'lane_centers': lane_centers,
        'scene_bounds': {'x': [0, scene_length], 'y': [-scene_width/2, scene_width/2]},
        'dt': dt
    }

# Generate synthetic scene
scene_data = generate_synthetic_scene()
print(f"✅ Generated scene with {len(scene_data['candidates'])} trajectory candidates")
print(f"   Candidates: {[name for name, _ in scene_data['candidates']]}")

## 3. Visualize the Scene

In [ ]:
def plot_scene(scene_data, title="Trajectory Candidates"):
    """Plot the synthetic scene with trajectory candidates."""
    
    fig, ax = plt.subplots(1, 1, figsize=(15, 8))
    
    # Plot lane boundaries
    bounds = scene_data['scene_bounds']
    lane_width = (bounds['y'][1] - bounds['y'][0]) / 3
    
    for i in range(4):  # 4 lane boundaries for 3 lanes
        y_pos = bounds['y'][0] + i * lane_width
        ax.axhline(y=y_pos, color='gray', linestyle='--', alpha=0.5, linewidth=1)
    
    # Plot lane centers
    for center in scene_data['lane_centers']:
        ax.axhline(y=center, color='gray', linestyle=':', alpha=0.3, linewidth=0.5)
    
    # Color map for trajectories
    colors = ['green', 'orange', 'red']
    
    # Plot trajectory candidates
    for i, (name, traj) in enumerate(scene_data['candidates']):
        ax.plot(traj[:, 0], traj[:, 1], 
                color=colors[i], linewidth=3, label=f'{name.title()} Trajectory', alpha=0.8)
        # Mark start and end points
        ax.scatter(traj[0, 0], traj[0, 1], color=colors[i], s=100, marker='o', zorder=5)
        ax.scatter(traj[-1, 0], traj[-1, 1], color=colors[i], s=100, marker='s', zorder=5)
    
    # Plot other agents
    for i, agent_traj in enumerate(scene_data['other_agents']):
        ax.plot(agent_traj[:, 0], agent_traj[:, 1], 
                color='blue', linewidth=2, linestyle='--', label=f'Other Agent {i+1}', alpha=0.6)
        ax.scatter(agent_traj[0, 0], agent_traj[0, 1], color='blue', s=80, marker='^', zorder=5)
    
    ax.set_xlabel('X Position (m)', fontsize=12)
    ax.set_ylabel('Y Position (m)', fontsize=12)
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.legend(loc='upper left')
    ax.grid(True, alpha=0.3)
    ax.set_aspect('equal')
    
    plt.tight_layout()
    plt.show()

plot_scene(scene_data)

## 4. Initialize Physics Checker

The physics checker evaluates trajectories based on kinematic constraints, collision risk, and comfort metrics.

In [ ]:
# Initialize physics checker
physics_checker = PhysicsChecker(
    collision_threshold=2.0,  # minimum distance to other agents (meters)
    max_acceleration=4.0,     # maximum acceleration (m/s²)
    max_jerk=3.0,            # maximum jerk (m/s³)
    comfort_weight=0.3,       # weight for comfort in overall score
    safety_weight=0.4,        # weight for safety in overall score
    progress_weight=0.3       # weight for progress in overall score
)

print("✅ Physics checker initialized")
print(f"   Collision threshold: {physics_checker.collision_threshold}m")
print(f"   Max acceleration: {physics_checker.max_acceleration}m/s²")
print(f"   Max jerk: {physics_checker.max_jerk}m/s³")

## 5. Evaluate Trajectories with Physics Checker

In [ ]:
def evaluate_trajectory_physics(trajectory, other_agents, dt, name):
    """Evaluate a single trajectory using physics-based metrics."""
    
    # Calculate kinematic properties
    positions = trajectory[:, :2]
    velocities = trajectory[:, 2:4]
    
    # Calculate accelerations
    accelerations = np.diff(velocities, axis=0) / dt
    
    # Calculate jerk
    jerks = np.diff(accelerations, axis=0) / dt
    
    # Collision checking
    min_distances = []
    for agent_traj in other_agents:
        distances = np.linalg.norm(positions - agent_traj[:len(positions), :2], axis=1)
        min_distances.append(np.min(distances))
    
    min_distance_to_agents = np.min(min_distances) if min_distances else float('inf')
    collision_risk = max(0, 1 - min_distance_to_agents / physics_checker.collision_threshold)
    
    # Comfort metrics
    max_acceleration = np.max(np.linalg.norm(accelerations, axis=1)) if len(accelerations) > 0 else 0
    max_jerk = np.max(np.linalg.norm(jerks, axis=1)) if len(jerks) > 0 else 0
    
    # Comfort scores (lower is better, so we invert)
    acceleration_comfort = max(0, 1 - max_acceleration / physics_checker.max_acceleration)
    jerk_comfort = max(0, 1 - max_jerk / physics_checker.max_jerk)
    comfort_score = (acceleration_comfort + jerk_comfort) / 2
    
    # Progress score (distance traveled)
    total_distance = np.sum(np.linalg.norm(np.diff(positions, axis=0), axis=1))
    progress_score = min(1.0, total_distance / 100.0)  # normalize by expected distance
    
    # Overall physics score
    safety_score = 1 - collision_risk
    overall_score = (
        physics_checker.safety_weight * safety_score +
        physics_checker.comfort_weight * comfort_score +
        physics_checker.progress_weight * progress_score
    )
    
    return {
        'name': name,
        'overall_score': overall_score,
        'safety_score': safety_score,
        'comfort_score': comfort_score,
        'progress_score': progress_score,
        'collision_risk': collision_risk,
        'min_distance': min_distance_to_agents,
        'max_acceleration': max_acceleration,
        'max_jerk': max_jerk,
        'total_distance': total_distance
    }

# Evaluate all trajectory candidates
physics_results = []
for name, trajectory in scene_data['candidates']:
    result = evaluate_trajectory_physics(
        trajectory, scene_data['other_agents'], scene_data['dt'], name
    )
    physics_results.append(result)

# Display results
print("🔬 Physics-Based Evaluation Results:")
print("=" * 60)

for result in physics_results:
    print(f"\n{result['name'].upper()} TRAJECTORY:")
    print(f"  Overall Score:     {result['overall_score']:.3f}")
    print(f"  Safety Score:      {result['safety_score']:.3f} (collision risk: {result['collision_risk']:.3f})")
    print(f"  Comfort Score:     {result['comfort_score']:.3f}")
    print(f"  Progress Score:    {result['progress_score']:.3f}")
    print(f"  Min Distance:      {result['min_distance']:.2f}m")
    print(f"  Max Acceleration:  {result['max_acceleration']:.2f}m/s²")
    print(f"  Max Jerk:          {result['max_jerk']:.2f}m/s³")

## 6. Visualize Physics Analysis

In [ ]:
def plot_physics_analysis(physics_results):
    """Create physics analysis plots."""\n    
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    # Extract data for plotting
    names = [r['name'].title() for r in physics_results]
    colors = ['green', 'orange', 'red']
    
    # 1. Overall Scores Comparison
    ax1 = axes[0, 0]
    overall_scores = [r['overall_score'] for r in physics_results]
    bars1 = ax1.bar(names, overall_scores, color=colors, alpha=0.7)
    ax1.set_title('Overall Physics Scores', fontweight='bold')
    ax1.set_ylabel('Score')
    ax1.set_ylim(0, 1)
    
    # Add value labels on bars
    for bar, score in zip(bars1, overall_scores):
        ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, 
                f'{score:.3f}', ha='center', va='bottom', fontweight='bold')
    
    # 2. Component Scores Breakdown
    ax2 = axes[0, 1]
    safety_scores = [r['safety_score'] for r in physics_results]
    comfort_scores = [r['comfort_score'] for r in physics_results]
    progress_scores = [r['progress_score'] for r in physics_results]
    
    x = np.arange(len(names))
    width = 0.25
    
    ax2.bar(x - width, safety_scores, width, label='Safety', alpha=0.8, color='blue')
    ax2.bar(x, comfort_scores, width, label='Comfort', alpha=0.8, color='green')
    ax2.bar(x + width, progress_scores, width, label='Progress', alpha=0.8, color='purple')
    
    ax2.set_title('Component Scores Breakdown', fontweight='bold')
    ax2.set_ylabel('Score')
    ax2.set_xticks(x)
    ax2.set_xticklabels(names)
    ax2.legend()
    ax2.set_ylim(0, 1)
    
    # 3. Safety Metrics
    ax3 = axes[1, 0]
    min_distances = [r['min_distance'] for r in physics_results]
    collision_risks = [r['collision_risk'] for r in physics_results]
    
    ax3_twin = ax3.twinx()
    
    bars3 = ax3.bar(names, min_distances, alpha=0.7, color='lightblue', label='Min Distance')
    line3 = ax3_twin.plot(names, collision_risks, 'ro-', linewidth=2, markersize=8, label='Collision Risk')
    
    ax3.axhline(y=2.0, color='red', linestyle='--', alpha=0.7, label='Safety Threshold')
    
    ax3.set_title('Safety Analysis', fontweight='bold')
    ax3.set_ylabel('Min Distance to Agents (m)', color='blue')
    ax3_twin.set_ylabel('Collision Risk', color='red')
    ax3.tick_params(axis='y', labelcolor='blue')
    ax3_twin.tick_params(axis='y', labelcolor='red')
    ax3.legend(loc='upper left')
    ax3_twin.legend(loc='upper right')
    
    # 4. Comfort Metrics
    ax4 = axes[1, 1]
    max_accelerations = [r['max_acceleration'] for r in physics_results]
    max_jerks = [r['max_jerk'] for r in physics_results]
    
    x = np.arange(len(names))
    width = 0.35
    
    bars4a = ax4.bar(x - width/2, max_accelerations, width, label='Max Acceleration', alpha=0.8, color='orange')
    
    ax4_twin = ax4.twinx()
    bars4b = ax4_twin.bar(x + width/2, max_jerks, width, label='Max Jerk', alpha=0.8, color='red')
    
    ax4.axhline(y=4.0, color='orange', linestyle='--', alpha=0.7)
    ax4_twin.axhline(y=3.0, color='red', linestyle='--', alpha=0.7)
    
    ax4.set_title('Comfort Analysis', fontweight='bold')
    ax4.set_ylabel('Max Acceleration (m/s²)', color='orange')
    ax4_twin.set_ylabel('Max Jerk (m/s³)', color='red')
    ax4.set_xticks(x)
    ax4.set_xticklabels(names)
    ax4.tick_params(axis='y', labelcolor='orange')
    ax4_twin.tick_params(axis='y', labelcolor='red')
    ax4.legend(loc='upper left')
    ax4_twin.legend(loc='upper right')
    
    plt.tight_layout()
    plt.show()

plot_physics_analysis(physics_results)

## 7. Initialize Neural Trajectory Critic

Now let's create and demonstrate a neural network-based trajectory critic.

In [ ]:
# Initialize trajectory critic model
critic_model = TrajectoryCritic(
    state_dim=32,      # dimension of agent state encoding
    lane_dim=64,       # dimension of lane graph encoding
    cand_dim=64,       # dimension of trajectory candidate encoding
    hidden_dim=128,    # hidden layer dimension
    dropout=0.1        # dropout rate
)

print("🧠 Neural Trajectory Critic initialized")
print(f"   Model parameters: {sum(p.numel() for p in critic_model.parameters()):,}")
print(f"   State encoding dim: {critic_model.state_dim}")
print(f"   Lane encoding dim: {critic_model.lane_dim}")
print(f"   Trajectory encoding dim: {critic_model.cand_dim}")

## 8. Simulate Neural Critic Scoring

Since we don't have a pre-trained model, we'll simulate the neural critic's behavior to demonstrate the framework.

In [ ]:
def simulate_neural_critic_scores(trajectories, physics_results):
    """Simulate neural critic scores based on trajectory features."""
    
    neural_results = []
    
    for i, (name, trajectory) in enumerate(trajectories):
        # Extract trajectory features
        positions = trajectory[:, :2]
        velocities = trajectory[:, 2:4]
        
        # Calculate feature-based scores (simulating learned patterns)
        
        # 1. Smoothness score (based on curvature)
        if len(positions) > 2:
            # Calculate path curvature
            dx = np.diff(positions[:, 0])
            dy = np.diff(positions[:, 1])
            d2x = np.diff(dx)
            d2y = np.diff(dy)
            
            curvature = np.abs(dx[:-1] * d2y - dy[:-1] * d2x) / (dx[:-1]**2 + dy[:-1]**2)**(3/2)
            curvature = curvature[~np.isnan(curvature)]  # remove NaN values
            avg_curvature = np.mean(curvature) if len(curvature) > 0 else 0
            smoothness_score = max(0, 1 - avg_curvature * 10)  # penalize high curvature
        else:
            smoothness_score = 1.0
        
        # 2. Consistency score (velocity consistency)
        speed = np.linalg.norm(velocities, axis=1)
        speed_variance = np.var(speed)
        consistency_score = max(0, 1 - speed_variance / 25)  # penalize speed variations
        
        # 3. Efficiency score (straight-line distance vs path length)
        if len(positions) > 1:
            straight_distance = np.linalg.norm(positions[-1] - positions[0])
            path_length = np.sum(np.linalg.norm(np.diff(positions, axis=0), axis=1))
            efficiency_score = straight_distance / path_length if path_length > 0 else 0
        else:
            efficiency_score = 1.0
        
        # 4. Combine with physics-based insights (simulating learned correlation)
        physics_score = physics_results[i]['overall_score']
        
        # Neural critic learns to weight different aspects
        neural_score = (
            0.3 * smoothness_score +
            0.2 * consistency_score +
            0.2 * efficiency_score +
            0.3 * physics_score  # incorporates physics knowledge
        )
        
        # Add some learned bias (simulating training on real data)
        if name == 'straight':
            neural_score += 0.1  # bias towards straight trajectories
        elif name == 'aggressive':
            neural_score -= 0.15  # strong bias against aggressive driving
        
        neural_score = max(0, min(1, neural_score))  # clamp to [0, 1]
        
        neural_results.append({
            'name': name,
            'neural_score': neural_score,
            'smoothness_score': smoothness_score,
            'consistency_score': consistency_score,
            'efficiency_score': efficiency_score,
            'physics_correlation': physics_score
        })
    
    return neural_results

# Generate neural critic scores
neural_results = simulate_neural_critic_scores(scene_data['candidates'], physics_results)

print("🧠 Neural Critic Evaluation Results:")
print("=" * 60)

for result in neural_results:
    print(f"\n{result['name'].upper()} TRAJECTORY:")
    print(f"  Neural Score:      {result['neural_score']:.3f}")
    print(f"  Smoothness:        {result['smoothness_score']:.3f}")
    print(f"  Consistency:       {result['consistency_score']:.3f}")
    print(f"  Efficiency:        {result['efficiency_score']:.3f}")
    print(f"  Physics Corr:      {result['physics_correlation']:.3f}")

## 9. Compare Physics vs Neural Evaluation

In [ ]:
def plot_comparison_analysis(physics_results, neural_results):
    """Compare physics-based and neural critic evaluations."""
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    
    names = [r['name'].title() for r in physics_results]
    colors = ['green', 'orange', 'red']
    
    # 1. Score Comparison
    ax1 = axes[0]
    physics_scores = [r['overall_score'] for r in physics_results]
    neural_scores = [r['neural_score'] for r in neural_results]
    
    x = np.arange(len(names))
    width = 0.35
    
    bars1 = ax1.bar(x - width/2, physics_scores, width, label='Physics-Based', alpha=0.8, color='blue')
    bars2 = ax1.bar(x + width/2, neural_scores, width, label='Neural Critic', alpha=0.8, color='purple')
    
    ax1.set_title('Physics vs Neural Critic Scores', fontweight='bold')
    ax1.set_ylabel('Score')
    ax1.set_xticks(x)
    ax1.set_xticklabels(names)
    ax1.legend()
    ax1.set_ylim(0, 1)
    
    # Add value labels
    for bar, score in zip(bars1, physics_scores):
        ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, 
                f'{score:.2f}', ha='center', va='bottom', fontsize=10)
    for bar, score in zip(bars2, neural_scores):
        ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, 
                f'{score:.2f}', ha='center', va='bottom', fontsize=10)
    
    # 2. Correlation Analysis
    ax2 = axes[1]
    ax2.scatter(physics_scores, neural_scores, c=colors, s=200, alpha=0.7)
    
    # Add trajectory labels
    for i, name in enumerate(names):
        ax2.annotate(name, (physics_scores[i], neural_scores[i]), 
                    xytext=(5, 5), textcoords='offset points', fontsize=10)
    
    # Add diagonal line for perfect correlation
    ax2.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Perfect Correlation')
    
    # Calculate and display correlation
    correlation = np.corrcoef(physics_scores, neural_scores)[0, 1]
    ax2.text(0.05, 0.95, f'Correlation: {correlation:.3f}', 
            transform=ax2.transAxes, fontsize=12, fontweight='bold',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
    ax2.set_xlabel('Physics-Based Score')
    ax2.set_ylabel('Neural Critic Score')
    ax2.set_title('Score Correlation Analysis', fontweight='bold')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    ax2.set_xlim(0, 1)
    ax2.set_ylim(0, 1)
    
    # 3. Ranking Comparison
    ax3 = axes[2]
    
    # Get rankings (1 = best, 3 = worst)
    physics_ranks = [sorted(physics_scores, reverse=True).index(score) + 1 for score in physics_scores]
    neural_ranks = [sorted(neural_scores, reverse=True).index(score) + 1 for score in neural_scores]
    
    # Create ranking comparison table
    ranking_data = []
    for i, name in enumerate(names):
        ranking_data.append([name, physics_ranks[i], neural_ranks[i]])
    
    # Plot as table
    ax3.axis('tight')
    ax3.axis('off')
    
    table = ax3.table(cellText=ranking_data,
                     colLabels=['Trajectory', 'Physics Rank', 'Neural Rank'],
                     cellLoc='center',
                     loc='center')
    
    table.auto_set_font_size(False)
    table.set_fontsize(12)
    table.scale(1.2, 1.5)
    
    # Color code the rankings
    for i in range(len(names)):
        # Color trajectory names
        table[(i+1, 0)].set_facecolor(colors[i])
        table[(i+1, 0)].set_text_props(weight='bold', color='white')
        
        # Color rankings based on agreement
        if physics_ranks[i] == neural_ranks[i]:
            # Perfect agreement - green
            table[(i+1, 1)].set_facecolor('lightgreen')
            table[(i+1, 2)].set_facecolor('lightgreen')
        else:
            # Disagreement - light red
            table[(i+1, 1)].set_facecolor('lightcoral')
            table[(i+1, 2)].set_facecolor('lightcoral')
    
    ax3.set_title('Ranking Comparison\n(Green = Agreement, Red = Disagreement)', 
                 fontweight='bold', pad=20)
    
    plt.tight_layout()
    plt.show()

plot_comparison_analysis(physics_results, neural_results)

## 10. Summary and Key Insights

In [ ]:
def print_summary(physics_results, neural_results):
    """Print a summary of the evaluation."""\n    
    print("📊 EVALUATION SUMMARY")
    print("=" * 80)
    
    # Best trajectory according to each method
    physics_best = max(physics_results, key=lambda x: x['overall_score'])
    neural_best = max(neural_results, key=lambda x: x['neural_score'])
    
    print(f"\n🏆 BEST TRAJECTORY SELECTION:")
    print(f"   Physics-Based:  {physics_best['name'].title()} (score: {physics_best['overall_score']:.3f})")
    print(f"   Neural Critic:  {neural_best['name'].title()} (score: {neural_best['neural_score']:.3f})")
    
    agreement = physics_best['name'] == neural_best['name']
    print(f"   Agreement:      {'✅ YES' if agreement else '❌ NO'}")
    
    # Score correlation
    physics_scores = [r['overall_score'] for r in physics_results]
    neural_scores = [r['neural_score'] for r in neural_results]
    correlation = np.corrcoef(physics_scores, neural_scores)[0, 1]
    
    print(f"\n📈 CORRELATION ANALYSIS:")
    print(f"   Score Correlation:  {correlation:.3f}")
    if correlation > 0.8:
        print(f"   Interpretation:     Strong positive correlation - methods agree well")
    elif correlation > 0.5:
        print(f"   Interpretation:     Moderate positive correlation - some agreement")
    else:
        print(f"   Interpretation:     Weak correlation - methods disagree significantly")
    
    # Key insights
    print(f"\n🔍 KEY INSIGHTS:")
    
    # Safety analysis
    safest_physics = min(physics_results, key=lambda x: x['collision_risk'])
    print(f"   Safest Trajectory:  {safest_physics['name'].title()} (min distance: {safest_physics['min_distance']:.2f}m)")
    
    # Comfort analysis
    most_comfortable = max(physics_results, key=lambda x: x['comfort_score'])
    print(f"   Most Comfortable:   {most_comfortable['name'].title()} (comfort: {most_comfortable['comfort_score']:.3f})")
    
    # Efficiency analysis
    most_efficient = max(physics_results, key=lambda x: x['progress_score'])
    print(f"   Most Efficient:     {most_efficient['name'].title()} (progress: {most_efficient['progress_score']:.3f})")
    
    # Neural insights
    smoothest_neural = max(neural_results, key=lambda x: x['smoothness_score'])
    print(f"   Smoothest Path:     {smoothest_neural['name'].title()} (smoothness: {smoothest_neural['smoothness_score']:.3f})")
    
    print(f"\n💡 RECOMMENDATIONS:")
    
    if agreement:
        print(f"   ✅ Both methods agree on the best trajectory ({physics_best['name'].title()})")
        print(f"   ✅ This suggests robust evaluation across different criteria")
    else:
        print(f"   ⚠️  Methods disagree on best trajectory - consider hybrid approach")
        print(f"   ⚠️  Physics favors safety/comfort, Neural may capture learned preferences")
    
    # Identify problematic trajectories
    worst_physics = min(physics_results, key=lambda x: x['overall_score'])
    if worst_physics['collision_risk'] > 0.5:
        print(f"   🚨 High collision risk detected in {worst_physics['name'].title()} trajectory")
    
    if any(r['max_acceleration'] > 4.0 for r in physics_results):
        high_accel = [r for r in physics_results if r['max_acceleration'] > 4.0]
        print(f"   ⚠️  Excessive acceleration in: {', '.join(r['name'].title() for r in high_accel)}")
    
    print(f"\n🎯 NEXT STEPS:")
    print(f"   1. Train neural critic on real trajectory data")
    print(f"   2. Validate physics constraints with vehicle dynamics")
    print(f"   3. Collect human preference data for ground truth")
    print(f"   4. Implement real-time trajectory selection system")
    print(f"   5. Test in closed-loop simulation environment")

print_summary(physics_results, neural_results)

## 11. Next Steps and Advanced Usage

This notebook demonstrated the core concepts of PlanCritic. Here are some next steps to explore:

### 🚀 Advanced Tutorials
- **Training Custom Models**: See `02_training.ipynb` for training neural critics on real data
- **Physics Analysis**: See `03_physics_analysis.ipynb` for detailed physics constraint analysis
- **Web Visualization**: See `04_visualization.ipynb` for interactive trajectory visualization

### 🔧 Real-World Integration
```python
# Example: Integration with planning system
from plancritic.cli.score import TrajectoryScorer

# Load trained model
scorer = TrajectoryScorer(model_path="path/to/trained_model.pth")

# Score trajectory candidates in real-time
def select_best_trajectory(candidates, scene_context):
    scores = scorer.score_batch(candidates)
    return candidates[np.argmax(scores)]
```

### 📊 Data Integration
```python
# Example: Load real WOMD data
from plancritic.data.adapters import WOMDAdapter

adapter = WOMDAdapter(data_path="/path/to/womd")
scene_data = adapter.load_scene("scene_12345")
```

### 🎮 Web Visualization
```bash
# Start interactive web viewer
cd ../web
npm run dev
```

### 🧪 Custom Physics Constraints
```python
# Example: Add custom physics check
class CustomPhysicsChecker(PhysicsChecker):
    def check_lane_keeping(self, trajectory, lane_boundaries):
        # Implement custom lane-keeping constraint
        return constraint_score
```

## 🎉 Congratulations!

You've successfully completed the PlanCritic getting started tutorial! You now understand:

- ✅ How to generate and visualize trajectory scenarios
- ✅ Physics-based trajectory evaluation metrics
- ✅ Neural network-based trajectory criticism
- ✅ Comparison and correlation analysis between methods
- ✅ Integration patterns for real-world usage

**Ready to dive deeper?** Check out the other notebooks in this series:
- `02_training.ipynb` - Train your own trajectory critics
- `03_physics_analysis.ipynb` - Deep dive into physics constraints
- `04_visualization.ipynb` - Interactive web-based visualization

**Questions or feedback?** Open an issue on our [GitHub repository](https://github.com/your-org/plancritic) or join the discussion!